# 5. RAG: answer questions from rulings

Ask a question in any language. The system finds the most relevant passages in a set of Dutch rulings and lets Gemini answer from those passages only, in the language of the question, citing each ruling by ECLI. The examples and the evaluation use English.

Pipeline (code in `../rag/`):

1. Translate the question to Dutch (Gemini API).
2. Vector search with `multilingual-e5-base`: 30 candidate passages from Chroma.
3. Rerank each (question, passage) pair with a cross-encoder. Keep passages with probability at least 0.5, at most 5, at most 2 per ruling.
4. Gemini answers from the kept passages, with the section of each passage (court's assessment, party argument, ...) in the prompt.
5. Refuse if no passage is kept, or if the draft answer says the passages do not address the question.

Nothing is trained here. All models are used as they are.

Before running: `rag/build_corpus.py`, `rag/chunk.py` and `rag/build_index.py` have been run once, and `python -m rag.evaluate` has written `../results/rag_answers.jsonl`. This notebook makes no Gemini calls: it reads the Dutch queries and answers recorded by the evaluation and runs retrieval locally. To ask a new question, use `python -m rag.answer "..."`.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import config
from rag.retrieve import Retriever

for path in [config.RAG_CORPUS, config.RAG_CHUNKS, config.RAG_INDEX_DIR]:
    print(path.relative_to(ROOT), "exists:", path.exists())

summary = json.loads((ROOT / "results" / "rag_eval.json").read_text(encoding="utf-8"))
records = pd.read_json(ROOT / "results" / "rag_answers.jsonl", lines=True)
by_id = records.set_index("id")

C:\Users\nlwen\Desktop\hbo-corp-justid\justid-machinelearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


data\rag\corpus.jsonl exists: True
data\rag\chunks.jsonl exists: True
data\rag\chroma exists: True


## Corpus and chunks

2 500 rulings from `test` and `natural_test`. Twelve topics are defined by a keyword rule plus a rechtsgebied (see `build_corpus.py`). The rest are random rulings with no topic, so retrieval has something to get wrong. Rulings that mention a refusal subject (flight delays, crypto, ...) were left out on purpose.

In [2]:
corpus = pd.read_json(config.RAG_CORPUS, lines=True)
print("rulings:", len(corpus))
print("no topic (distractors):", int((corpus["topics"].map(len) == 0).sum()))
corpus["topics"].explode().value_counts()

rulings: 2500
no topic (distractors): 1618


topics
environmental_permit    100
bijstand_recovery       100
tenancy                 100
tbs                     100
debt_restructuring       81
wia                      75
contact_arrangement      70
woz_value                61
child_support            59
asylum_credibility       51
drink_driving            49
summary_dismissal        43
Name: count, dtype: int64

Each ruling is cut into passages of about 250 words with 40 words of overlap. A passage never crosses a section heading, so every passage has one section label.

In [3]:
chunks = pd.read_json(config.RAG_CHUNKS, lines=True)
print("chunks:", len(chunks))
print("words per chunk, median:", int(chunks["text"].str.split().map(len).median()))
chunks["section"].value_counts(normalize=True).round(3)

chunks: 44490


words per chunk, median: 200


section
court        0.519
procedure    0.143
parties      0.104
decision     0.079
facts        0.072
header       0.063
advisory     0.019
Name: proportion, dtype: float64

## Retrieval in two stages

Vector scores from e5 sit in a narrow band (about 0.85 to 0.89) for relevant and irrelevant passages alike, so they cannot decide what to keep. The cross-encoder reads question and passage together and gives a probability that separates them much better.

In [4]:
retriever = Retriever()

rec = by_id.loc["q09"]
question, dutch = rec["question"], rec["dutch_query"]
print("Question:", question)
print("Dutch query:", dutch)

stage1 = sorted(retriever.candidates([dutch, question]), key=lambda h: h.score, reverse=True)
pd.DataFrame([{"ecli": h.ecli, "section": h.section, "vector": round(h.score, 3)} for h in stage1[:5]])

Question: How do courts decide whether the WOZ value of a home was set too high?
Dutch query: Hoe beoordeelt de rechter of de WOZ-waarde van een woning te hoog is vastgesteld?


,ecli,section,vector
0,ECLI:NL:RBMNE:2021:5143,parties,0.898
1,ECLI:NL:RBZWB:2025:1007,court,0.896
2,ECLI:NL:RBZWB:2024:6439,court,0.896
3,ECLI:NL:GHDHA:2024:2680,court,0.895
4,ECLI:NL:RBZWB:2021:4426,court,0.895


In [5]:
stage2 = retriever.search(question, dutch)
pd.DataFrame([{"ecli": h.ecli, "section": h.section, "vector": round(h.score, 3), "rerank": round(h.rerank, 3)} for h in stage2])

,ecli,section,vector,rerank
0,ECLI:NL:RBZWB:2024:6439,court,0.896,1.0
1,ECLI:NL:RBZWB:2024:300,court,0.894,1.0
2,ECLI:NL:RBZWB:2024:6332,court,0.892,1.0
3,ECLI:NL:RBZWB:2025:1007,court,0.896,1.0
4,ECLI:NL:RBZWB:2024:8603,court,0.887,1.0


## Full answer

In [6]:
print(rec["answer"])
print()
for s in rec["sources"]:
    print(f"{s['ecli']}  [{s['section']}]  rerank {s['rerank']:.2f}")
print("citations outside the passages:", rec["invalid_citations"])

The court assesses whether the value of the home has been set too high based on the arguments submitted by the interested party, which are the grounds of appeal [ECLI:NL:RBZWB:2024:6439, ECLI:NL:RBZWB:2024:300, ECLI:NL:RBZWB:2024:6332, ECLI:NL:RBZWB:2025:1007, ECLI:NL:RBZWB:2024:8603]. Under Article 17, second paragraph, of the Wet WOZ, the value of the home is determined at the value to be assigned if the full and unencumbered ownership could be transferred and the acquirer could immediately and fully use the property in its current state [ECLI:NL:RBZWB:2024:300, ECLI:NL:RBZWB:2024:6332, ECLI:NL:RBZWB:2025:1007]. According to the legislature's intent, this value is "the price that would be spent by the highest bidder upon offering for sale in the most suitable manner after the best preparation" [ECLI:NL:RBZWB:2024:300, ECLI:NL:RBZWB:2024:6332, ECLI:NL:RBZWB:2025:1007].

ECLI:NL:RBZWB:2024:6439  [court]  rerank 1.00
ECLI:NL:RBZWB:2024:300  [court]  rerank 1.00
ECLI:NL:RBZWB:2024:6332  

## Refusal

Two ways to refuse. Crypto: no passage passes the reranker, so the LLM is never called. Solar panels: some passages pass (roof repairs, "solartubes" skylights), Gemini writes a draft, and the draft itself says the rulings do not cover the question.

In [7]:
for qid in ["r05", "r03"]:
    r = by_id.loc[qid]
    print(r["question"])
    print("  passages kept:", len(r["sources"]))
    print("  draft:", r["draft"][:300] if r["draft"] else "(LLM not called)")
    print("  final:", r["answer"])
    print()

How is cryptocurrency taxed in box 3?
  passages kept: 0
  draft: (LLM not called)
  final: Not found in the provided rulings.

Can a homeowner be forced to remove solar panels from their roof?
  passages kept: 4
  draft: The provided passages do not contain information about whether a homeowner can be forced to remove solar panels from their roof.
  final: Not found in the provided rulings.



## Evaluation

`python -m rag.evaluate` runs all 29 questions in `../rag/eval/questions.jsonl` (24 topic questions, 5 that must be refused) and writes `../results/rag_eval.json` and `../results/rag_answers.jsonl`.

- **retrieval_hit_rate**: at least one kept passage comes from a ruling with the question's topic
- **retrieval_precision**: share of kept passages from on-topic rulings
- **refusal_correct**: refusal questions that were refused
- **invalid_citation_rate**: cited ECLIs that are not among the passages given to the LLM

In [8]:
pd.Series({k: v for k, v in summary.items() if k != "settings"})

questions                29.000000
topical_questions        24.000000
refusal_questions         5.000000
retrieval_hit_rate        1.000000
retrieval_precision       0.950000
topical_answered          0.916667
refusal_correct           1.000000
answers_with_citation     1.000000
invalid_citation_rate     0.000000
median_seconds            4.700000
dtype: float64

In [9]:
records["kept"] = records["sources"].map(len)
records["on_topic"] = records["sources"].map(lambda s: sum(bool(x["on_topic"]) for x in s))
records["cited"] = records["cited"].map(len)
records[["id", "topic", "refused", "kept", "on_topic", "cited", "seconds"]]

,id,topic,refused,kept,on_topic,cited,seconds
0,q01,tenancy,True,5,4,0,8.0
1,q02,tenancy,True,5,5,0,5.4
2,q03,summary_dismissal,False,5,5,3,5.0
3,q04,summary_dismissal,False,5,5,3,4.2
4,q05,bijstand_recovery,False,5,5,4,4.5
5,q06,bijstand_recovery,False,5,5,3,4.9
6,q07,asylum_credibility,False,5,5,3,4.9
7,q08,asylum_credibility,False,5,5,3,4.3
8,q09,woz_value,False,5,5,5,39.7
9,q10,woz_value,False,5,5,5,4.4


## Manual check

A valid ECLI does not mean a correct claim. Three answers from the evaluation run, each shown next to the passages it was given. Retrieval is deterministic, so searching again with the recorded Dutch query returns the same passages.

In [10]:
def show(qid: str) -> None:
    rec = by_id.loc[qid]
    hits = retriever.search(rec["question"], rec["dutch_query"])
    assert [h.ecli for h in hits] == [s["ecli"] for s in rec["sources"]], "passages differ from the eval run"
    print(rec["question"])
    print()
    print(rec["answer"])
    for h in hits:
        print(f"\n--- {h.ecli} [{h.section}]")
        print(h.text)

In [11]:
show("q19")

Under what conditions do courts impose TBS with compulsory treatment?

Courts may impose the measure of TBS (terbeschikkingstelling) when statutory conditions are met, including that the defendant had a defective development or pathological mental disorder at the time of committing the offense [ECLI:NL:GHSHE:2023:2992]. Furthermore, the committed offense must be a crime carrying a statutory prison sentence of four years or more [ECLI:NL:GHSHE:2023:2992]. A court also requires advice from at least two behavioral experts from different disciplines, including a psychiatrist, who have examined the defendant [ECLI:NL:GHSHE:2023:2992]. Additionally, the safety of others or the general safety of persons or property must demand the imposition of the measure [ECLI:NL:RBDHA:2017:11127, ECLI:NL:GHSHE:2023:2992], and less intrusive measures must not be effective [ECLI:NL:RBDHA:2017:11127]. For compulsory treatment specifically, courts consider that safety requires state nursing, that alternatives 

In [12]:
show("q14")

When can child support be reduced because the paying parent's income has dropped?

Pursuant to Article 1:401, paragraph 1, of the Civil Code, a judicial decision or maintenance agreement can be modified or revoked if it subsequently ceases to meet statutory standards due to a change in circumstances [ECLI:NL:GHARL:2021:10432, ECLI:NL:GHARL:2021:9926]. The man argued that he had no capacity left to pay child support because he would have no employment income starting August 1, 2021, and would instead receive a sickness benefits agency payment of an unknown amount [ECLI:NL:GHARL:2021:10432].

--- ECLI:NL:GHARL:2021:10432 [court]
Kinderalimentatie Wijziging van omstandigheden 5.1 Ingevolge artikel 1:401 lid 1 van het van het Burgerlijk Wetboek kan een rechterlijke uitspraak of een overeenkomst betreffende levensonderhoud bij latere rechterlijke uitspraak worden gewijzigd of ingetrokken, wanneer zij nadien door wijziging van omstandigheden ophoudt aan de wettelijke maatstaven te voldoen. D

In [13]:
show("q11")

When do courts restrict or refuse contact between a parent and a child?

Under Article 1:377a paragraph 3 of the Civil Code, a court denies the right to contact between a parent and a child only if contact would cause serious harm to the child's mental or physical development, the parent is manifestly unsuitable or unable to handle contact, a child aged twelve or older expresses serious objections to contact during a hearing, or contact is otherwise contrary to the child's weighty interests [ECLI:NL:GHAMS:2022:1315, ECLI:NL:GHARL:2018:10854]. Additionally, a court may decide not to bring about forced contact between a child and a parent when it is not in the child's best interests, granting the child time to take the step toward contact independently [ECLI:NL:GHARN:2011:BQ5048].

--- ECLI:NL:RBZWB:2024:6687 [court]
het daarom ook enorm moeilijk om zijn vader te treffen in de supermarkt waar hij op dat moment aan het werk was. Na het voorval in de supermarkt is de man buiten zinnen van 

### Verdict

Each claim checked against the passage it cites:

- **q19 (TBS)**: 4 of 5 correct, 1 partly. The conditions of Article 37a Sr, the two-expert advice, and why compulsory treatment was chosen (refusal to cooperate, only TBS gives long-term treatment) match the cited rulings. The last sentence also cites ECLI:NL:RBDHA:2017:11127 for compulsory treatment specifically, but that passage is about TBS in general.
- **q14 (child support)**: 2 of 2 correct, but incomplete. Article 1:401 BW is stated correctly, and the man's claim of lost income is presented as his argument. The answer misses ECLI:NL:GHLEE:2011:BR0590, where the court did set child support to nil because the man's income was too low, which is the most direct answer among the passages.
- **q11 (contact)**: 2 of 2 correct. The four grounds of Article 1:377a(3) BW and the suspension of contact in ECLI:NL:GHARN:2011:BQ5048 match the passages. The party arguments in ECLI:NL:GHSHE:2015:2229 are not used.

Limits: 29 questions is a small set, and the answer prompt was adjusted after seeing false refusals on some of these questions, so they are not fully held out. Gemini's translation differs slightly between runs even at temperature 0, which changes the passages; between two runs, q01 and q02 flipped from answered to refused. A correct claim also does not mean a complete answer: the system only knows what its five passages say, and does not always use the best one.